In [ ]:
import time, os, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

DRIVE_TRAIN     = "/content/drive/MyDrive/debris-detection/train"
DRIVE_VAL       = "/content/drive/MyDrive/debris-detection/val"
DRIVE_TRAIN_CSV = "/content/drive/MyDrive/debris-detection/train.csv"
DRIVE_VAL_CSV   = "/content/drive/MyDrive/debris-detection/val.csv"

LOCAL_TRAIN     = "/content/train"
LOCAL_VAL       = "/content/val"
LOCAL_TRAIN_CSV = "/content/train.csv"
LOCAL_VAL_CSV   = "/content/val.csv"

NUM_THREADS = 16


def parallel_copy(src_dir, dst_dir, label=""):
    os.makedirs(dst_dir, exist_ok=True)
    existing = set(os.listdir(dst_dir))

    tasks = []
    skipped = 0
    for f in os.listdir(src_dir):
        src = os.path.join(src_dir, f)
        dst = os.path.join(dst_dir, f)
        if not os.path.isfile(src):
            continue
        if f in existing and os.path.getsize(dst) == os.path.getsize(src):
            skipped += 1
        else:
            tasks.append((src, dst))

    if skipped:
        print(f"   Skipped {skipped} already-copied files")

    if not tasks:
        print(f"  All files already present")
        return len(existing)

    failed = []
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as ex:
        futures = {ex.submit(shutil.copy2, s, d): s for s, d in tasks}
        with tqdm(total=len(tasks), desc=label, unit="file") as pbar:
            for future in as_completed(futures):
                try:
                    future.result()
                except Exception as e:
                    failed.append((futures[future], str(e)))
                pbar.update(1)

    if failed:
        print(f"   Error: {len(failed)} failed")

    return len(os.listdir(dst_dir))


t0 = time.time()

print("Copying train images...")
n_train = parallel_copy(DRIVE_TRAIN, LOCAL_TRAIN, "Train")

print("Copying val images...")
n_val = parallel_copy(DRIVE_VAL, LOCAL_VAL, "Val")

print("Copying CSVs...")
shutil.copy2(DRIVE_TRAIN_CSV, LOCAL_TRAIN_CSV)
shutil.copy2(DRIVE_VAL_CSV, LOCAL_VAL_CSV)

print(f"\nDone in {(time.time()-t0)/60:.1f} min — Train: {n_train}, Val: {n_val}")

In [ ]:
!pip install -q albumentations pycocotools tqdm tensorboard

import torch
!nvidia-smi
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16: {torch.cuda.is_bf16_supported()}")

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PROJECT = "/content/drive/MyDrive/debris-detection"
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print(f"Drive mounted -> {DRIVE_PROJECT}")

In [ ]:
import os, ast, csv, json, time, logging, warnings
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm

import torch
import torch.utils.data as data
import torchvision
from torchvision import transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

class Config:
    TRAIN_DIR = "/content/train"
    CSV_PATH = "/content/train.csv"
    OUTPUT_DIR = "/content/outputs"
    CHECKPOINT_DIR = "/content/outputs/checkpoints"
    LOG_DIR = "/content/outputs/logs"
    TENSORBOARD_DIR = "/content/outputs/tensorboard"
    DRIVE_BACKUP_DIR = "/content/drive/MyDrive/space_debris_detection"

    IMG_SIZE = 512
    IMG_FORMAT = ".jpg"
    NUM_CLASSES = 2
    BACKBONE = "resnet50"
    PRETRAINED = True

    BATCH_SIZE = 16
    NUM_WORKERS = 8
    PREFETCH_FACTOR = 4
    LEARNING_RATE = 0.01
    MOMENTUM = 0.9
    WEIGHT_DECAY = 0.0005
    NUM_EPOCHS = 50
    WARMUP_EPOCHS = 3
    LR_STEP_SIZE = 15
    LR_GAMMA = 0.1

    VAL_SPLIT = 0.15
    CONFIDENCE_THRESHOLD = 0.5
    NMS_THRESHOLD = 0.3
    TARGET_CONFIDENCE = 0.90

    SAVE_EVERY_N_EPOCHS = 5
    EARLY_STOP_PATIENCE = 10
    SEED = 42

    DEVICE = torch.device("cuda")
    USE_AMP = True
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

config = Config()

for d in [config.OUTPUT_DIR, config.CHECKPOINT_DIR, config.LOG_DIR,
          config.TENSORBOARD_DIR, config.DRIVE_BACKUP_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"{torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB | AMP: {config.AMP_DTYPE} | BS: {config.BATCH_SIZE}")

In [ ]:
from torch.utils.tensorboard import SummaryWriter

DRIVE_LOG_DIR = os.path.join(config.DRIVE_BACKUP_DIR, "logs")
DRIVE_TB_DIR = os.path.join(config.DRIVE_BACKUP_DIR, "tensorboard")
os.makedirs(DRIVE_LOG_DIR, exist_ok=True)
os.makedirs(DRIVE_TB_DIR, exist_ok=True)

log_file = os.path.join(DRIVE_LOG_DIR, f"training_{datetime.now():%Y%m%d_%H%M%S}.log")

logger = logging.getLogger("SpaceDebrisDetector")
logger.setLevel(logging.DEBUG)
logger.handlers = []

ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
ch.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S"))
logger.addHandler(ch)

fh = logging.FileHandler(log_file)
fh.setLevel(logging.DEBUG)
fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(name)s — %(message)s"))
logger.addHandler(fh)

tb_writer = SummaryWriter(log_dir=DRIVE_TB_DIR)

metrics_csv = os.path.join(DRIVE_LOG_DIR, "metrics.csv")
with open(metrics_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch", "train_loss", "train_loss_classifier", "train_loss_box_reg",
        "train_loss_objectness", "train_loss_rpn_box_reg",
        "val_mAP_50", "val_mAP_75", "val_avg_confidence",
        "val_detections_above_90pct", "learning_rate", "epoch_time_sec",
        "gpu_mem_allocated_gb", "gpu_mem_reserved_gb"
    ])

print(f"Logging: {log_file}")
print(f"Metrics: {metrics_csv}")
print(f"TensorBoard: {DRIVE_TB_DIR}")

In [ ]:
import random

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(config.SEED)
logger.info(f"Seed: {config.SEED}")
print(f"Seed: {config.SEED}")

In [ ]:
df = pd.read_csv(config.CSV_PATH)
print(f"CSV: {df.shape}, columns: {list(df.columns)}")
print(df.head(3))

col_map = {}
for col in df.columns:
    cl = col.strip().lower()
    if "imageid" in cl or "image_id" in cl or cl == "id":
        col_map[col] = "ImageId"
    elif "bbox" in cl or "box" in cl:
        col_map[col] = "bboxes"
df.rename(columns=col_map, inplace=True)
assert "ImageId" in df.columns, f"No ImageId column. Found: {list(df.columns)}"
assert "bboxes" in df.columns, f"No bboxes column. Found: {list(df.columns)}"

def safe_parse_bboxes(bbox_str):
    try:
        if isinstance(bbox_str, list):
            return bbox_str
        parsed = ast.literal_eval(str(bbox_str))
        if isinstance(parsed, list):
            if len(parsed) > 0 and not isinstance(parsed[0], list):
                parsed = [parsed]
            return parsed
        return []
    except (ValueError, SyntaxError):
        return []

df["parsed_bboxes"] = df["bboxes"].apply(safe_parse_bboxes)

total_images = len(df)
total_boxes = df["parsed_bboxes"].apply(len).sum()
empty_boxes = (df["parsed_bboxes"].apply(len) == 0).sum()
avg_boxes = df["parsed_bboxes"].apply(len).mean()

image_files = sorted([f for f in os.listdir(config.TRAIN_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

if image_files:
    sample_img = Image.open(os.path.join(config.TRAIN_DIR, image_files[0]))
    print(f"Sample: {image_files[0]}, size={sample_img.size}, mode={sample_img.mode}")

print(f"\n{total_images} images, {total_boxes} boxes, avg {avg_boxes:.1f}/img, {empty_boxes} empty, {len(image_files)} files on disk")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Boxes per image
box_counts = df["parsed_bboxes"].apply(len)
axes[0, 0].hist(box_counts, bins=range(0, box_counts.max() + 2), edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title("Bounding Boxes per Image")
axes[0, 0].set_xlabel("Number of Boxes")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].axvline(box_counts.mean(), color='red', linestyle='--', label=f'Mean: {box_counts.mean():.1f}')
axes[0, 0].legend()

# Box dimensions
widths, heights = [], []
for bboxes in df["parsed_bboxes"]:
    for box in bboxes:
        widths.append(box[1] - box[0])
        heights.append(box[3] - box[2])

# Width vs height
axes[0, 1].hist2d(widths, heights, bins=30, cmap='YlOrRd')
axes[0, 1].set_title("Bounding Box Size Distribution")
axes[0, 1].set_xlabel("Width (pixels)")
axes[0, 1].set_ylabel("Height (pixels)")
plt.colorbar(axes[0, 1].collections[0], ax=axes[0, 1])

# Box centers
cx_list, cy_list = [], []
for bboxes in df["parsed_bboxes"]:
    for box in bboxes:
        cx_list.append((box[0] + box[1]) / 2)
        cy_list.append((box[2] + box[3]) / 2)

# Center heatmap
axes[1, 0].hist2d(cx_list, cy_list, bins=50, cmap='hot')
axes[1, 0].set_title("Box Center Heatmap (512x512)")
axes[1, 0].set_xlabel("X center")
axes[1, 0].set_ylabel("Y center")
axes[1, 0].set_xlim(0, 512)
axes[1, 0].set_ylim(0, 512)
axes[1, 0].invert_yaxis()

# Box areas
areas = [w * h for w, h in zip(widths, heights)]
axes[1, 1].hist(areas, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1, 1].set_title("Bounding Box Area Distribution")
axes[1, 1].set_xlabel("Area (pixels squared)")
axes[1, 1].set_ylabel("Frequency")
axes[1, 1].axvline(np.mean(areas), color='red', linestyle='--', label=f'Mean: {np.mean(areas):.0f}')
axes[1, 1].legend()

plt.suptitle("Space Debris Bounding Box Analysis", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "bbox_analysis.png"), dpi=150)
plt.show()

print(f"Width: {np.mean(widths):.1f}+/-{np.std(widths):.1f}, Height: {np.mean(heights):.1f}+/-{np.std(heights):.1f}, Area: {np.mean(areas):.0f}+/-{np.std(areas):.0f}")

In [ ]:
import re
from collections import Counter

def check_dupes(directory, label=""):
    files = sorted([f for f in os.listdir(directory) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    copy_dupes = [f for f in files if re.search(r'\(\d+\)', f)]

    counts = Counter(files)
    exact_dupes = {k: v for k, v in counts.items() if v > 1}

    print(f"{label}: {len(files)} files")
    if copy_dupes:
        print(f"  WARNING: {len(copy_dupes)} copy-dupes found:")
        for f in copy_dupes[:10]:
            print(f"    {f}")
    if exact_dupes:
        print(f"  WARNING: {len(exact_dupes)} exact duplicates found:")
        for f, n in list(exact_dupes.items())[:10]:
            print(f"    {f} x{n}")
    if not copy_dupes and not exact_dupes:
        print(f"  Clean — no duplicates found")

check_dupes(config.TRAIN_DIR, "TRAIN")
check_dupes("/content/val", "VAL")

In [ ]:
class SpaceDebrisDataset(data.Dataset):
    def __init__(self, dataframe, image_dir, transforms=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transforms = transforms
        self.is_train = is_train
        assert "filename" in self.df.columns, "DataFrame needs 'filename' column"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["filename"])
        image_np = np.array(Image.open(img_path).convert("RGB"))

        # [xmin, xmax, ymin, ymax] to [xmin, ymin, xmax, ymax]
        boxes = []
        for box in row["parsed_bboxes"]:
            xmin, xmax, ymin, ymax = box
            xmin = max(0, min(xmin, config.IMG_SIZE - 1))
            xmax = max(0, min(xmax, config.IMG_SIZE))
            ymin = max(0, min(ymin, config.IMG_SIZE - 1))
            ymax = max(0, min(ymax, config.IMG_SIZE))
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.ones((len(boxes),), dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

        if self.transforms is not None:
            if len(boxes) > 0:
                transformed = self.transforms(
                    image=image_np,
                    bboxes=boxes.numpy().tolist() if isinstance(boxes, torch.Tensor) else boxes,
                    labels=labels.numpy().tolist() if isinstance(labels, torch.Tensor) else labels
                )
            else:
                transformed = self.transforms(image=image_np, bboxes=[], labels=[])

            image_np = transformed["image"]
            t_boxes = transformed.get("bboxes", [])
            t_labels = transformed.get("labels", [])

            if len(t_boxes) > 0:
                boxes = torch.as_tensor(t_boxes, dtype=torch.float32)
                labels = torch.as_tensor(t_labels, dtype=torch.int64)
                area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            else:
                boxes = torch.zeros((0, 4), dtype=torch.float32)
                labels = torch.zeros((0,), dtype=torch.int64)
                area = torch.zeros((0,), dtype=torch.float32)

        if isinstance(image_np, np.ndarray):
            image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).float() / 255.0
        else:
            image_tensor = image_np.float() / 255.0 if image_np.max() > 1.0 else image_np.float()

        target = {
            "boxes": boxes,
            "labels": labels,
            "area": area,
            "image_id": torch.tensor([idx]),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }
        return image_tensor, target

    @staticmethod
    def collate_fn(batch):
        return tuple(zip(*batch))


# Training augmentations
def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
        A.GaussNoise(var_limit=(10, 50), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.3),
    ], bbox_params=A.BboxParams(
        format='pascal_voc', label_fields=['labels'], min_area=16, min_visibility=0.3
    ))

# Validation augmentations
def get_val_transforms():
    return A.Compose([], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

print("Dataset and augmentations defined")

In [ ]:
import re

VAL_DIR = "/content/val"
VAL_CSV = "/content/val.csv"

val_df = pd.read_csv(VAL_CSV)
col_map = {}
for col in val_df.columns:
    cl = col.strip().lower()
    if "imageid" in cl or "image_id" in cl or cl == "id":
        col_map[col] = "ImageId"
    elif "bbox" in cl or "box" in cl:
        col_map[col] = "bboxes"
val_df.rename(columns=col_map, inplace=True)
val_df["parsed_bboxes"] = val_df["bboxes"].apply(safe_parse_bboxes)

train_df = df.copy().sort_values("ImageId").reset_index(drop=True)
val_df = val_df.sort_values("ImageId").reset_index(drop=True)

train_files = sorted([f for f in os.listdir(config.TRAIN_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
val_files = sorted([f for f in os.listdir(VAL_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

def extract_numeric_id(filename):
    base = os.path.splitext(filename)[0]
    if base.isdigit():
        return int(base)
    match = re.search(r'(\d+)', base)
    return int(match.group(1)) if match else None

def build_id_to_file_map(file_list):
    return {extract_numeric_id(f): f for f in file_list if extract_numeric_id(f) is not None}

def map_df_to_files(dataframe, id_to_file, label):
    filenames = []
    missing = 0
    for _, row in dataframe.iterrows():
        img_id = int(row["ImageId"])
        if img_id in id_to_file:
            filenames.append(id_to_file[img_id])
        else:
            filenames.append(None)
            missing += 1
    dataframe["filename"] = filenames
    if missing:
        print(f"  {label}: {missing} ImageIds have no matching file")
        dataframe = dataframe[dataframe["filename"].notna()].reset_index(drop=True)
    return dataframe

train_id_to_file = build_id_to_file_map(train_files)
val_id_to_file = build_id_to_file_map(val_files)
train_df = map_df_to_files(train_df, train_id_to_file, "TRAIN")
val_df = map_df_to_files(val_df, val_id_to_file, "VAL")

print(f"Train: {len(train_df)} images, IDs {train_df['ImageId'].min()}..{train_df['ImageId'].max()}")
print(f"Val:   {len(val_df)} images, IDs {val_df['ImageId'].min()}..{val_df['ImageId'].max()}")

train_dataset = SpaceDebrisDataset(train_df, config.TRAIN_DIR, get_train_transforms(), is_train=True)
val_dataset = SpaceDebrisDataset(val_df, VAL_DIR, get_val_transforms(), is_train=False)

train_loader = data.DataLoader(
    train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
    num_workers=config.NUM_WORKERS, collate_fn=SpaceDebrisDataset.collate_fn,
    pin_memory=True, prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=True, drop_last=True
)
val_loader = data.DataLoader(
    val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
    num_workers=config.NUM_WORKERS, collate_fn=SpaceDebrisDataset.collate_fn,
    pin_memory=True, prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=True
)

images, targets = next(iter(train_loader))
val_images, val_targets = next(iter(val_loader))
print(f"\nTrain: {len(train_loader)} batches x {config.BATCH_SIZE}, shape: {images[0].shape}, boxes: {targets[0]['boxes'].shape}")
print(f"Val:   {len(val_loader)} batches x {config.BATCH_SIZE}, shape: {val_images[0].shape}, boxes: {val_targets[0]['boxes'].shape}")

In [ ]:
# Clear GPU memory 
import gc

# Delete model/optimizer/scaler
for var in ['model', 'optimizer', 'scaler', 'lr_scheduler']:
    if var in globals():
        del globals()[var]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"GPU memory after cleanup:")
print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"  Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
# Visualize a training batch with bounding boxes

def visualize_batch(images, targets, n=4, title="Training Samples"):
    n = min(n, len(images))
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]

    for i in range(n):
        img = np.clip(images[i].permute(1, 2, 0).cpu().numpy(), 0, 1)
        axes[i].imshow(img)

        boxes = targets[i]["boxes"].cpu().numpy()
        for box in boxes:
            xmin, ymin, xmax, ymax = box
            axes[i].add_patch(patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                linewidth=2, edgecolor='lime', facecolor='none'
            ))

        axes[i].set_title(f"Image {i} — {len(boxes)} boxes")
        axes[i].axis("off")

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, "sample_batch.png"), dpi=150)
    plt.show()

images, targets = next(iter(train_loader))
visualize_batch(images, targets, n=4, title="Training Batch with Ground Truth")

In [ ]:
# Build Faster R-CNN model

def build_model(num_classes=2, pretrained=True):
    model = fasterrcnn_resnet50_fpn(pretrained=pretrained)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.roi_heads.score_thresh = 0.05
    model.roi_heads.nms_thresh = config.NMS_THRESHOLD
    model.roi_heads.detections_per_img = 100
    return model

model = build_model(num_classes=config.NUM_CLASSES, pretrained=config.PRETRAINED)
model.to(config.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
torch.cuda.empty_cache()

print(f"Faster R-CNN ResNet-50-FPN on {torch.cuda.get_device_name(0)}")
print(f"  Params: {total_params:,} total, {trainable_params:,} trainable")
print(f"  GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, {torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

In [ ]:
# Optimizer, LR scheduler with warmup, GradScaler

# Different LR for backbone vs head
params = [
    {"params": [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad],
     "lr": config.LEARNING_RATE * 0.1},
    {"params": [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad],
     "lr": config.LEARNING_RATE},
]

optimizer = torch.optim.SGD(params, momentum=config.MOMENTUM, weight_decay=config.WEIGHT_DECAY)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.01, end_factor=1.0, total_iters=config.WARMUP_EPOCHS
)
main_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA
)
lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[config.WARMUP_EPOCHS]
)

scaler = torch.cuda.amp.GradScaler(enabled=config.USE_AMP)

print(f"SGD (LR={config.LEARNING_RATE}, backbone=0.1x), Warmup({config.WARMUP_EPOCHS}ep) + StepLR, AMP={config.USE_AMP} ({config.AMP_DTYPE})")

In [ ]:
# Evaluation functions — mAP and confidence analysis

@torch.no_grad()
def evaluate_model(model, data_loader, device, iou_threshold=0.5):
    model.eval()
    all_predictions, all_targets, all_confidences = [], [], []
    detections_above_90 = 0
    total_detections = 0

    for images, targets in tqdm(data_loader, desc="Evaluating", leave=False):
        images = [img.to(device, non_blocking=True) for img in images]
        with torch.cuda.amp.autocast(enabled=config.USE_AMP, dtype=config.AMP_DTYPE):
            outputs = model(images)

        for i, output in enumerate(outputs):
            pred_boxes = output["boxes"].cpu()
            pred_scores = output["scores"].cpu()
            pred_labels = output["labels"].cpu()
            gt_boxes = targets[i]["boxes"].cpu()
            gt_labels = targets[i]["labels"].cpu()

            all_predictions.append({"boxes": pred_boxes, "scores": pred_scores, "labels": pred_labels})
            all_targets.append({"boxes": gt_boxes, "labels": gt_labels})

            if len(pred_scores) > 0:
                all_confidences.extend(pred_scores.numpy().tolist())
                total_detections += len(pred_scores)
                detections_above_90 += (pred_scores >= 0.9).sum().item()

    map_50 = compute_map(all_predictions, all_targets, iou_threshold=0.5)
    map_75 = compute_map(all_predictions, all_targets, iou_threshold=0.75)
    map_90 = compute_map(all_predictions, all_targets, iou_threshold=0.9)
    avg_confidence = np.mean(all_confidences) if all_confidences else 0.0
    pct_above_90 = (detections_above_90 / total_detections * 100) if total_detections > 0 else 0.0

    return {
        "mAP_50": map_50,
        "mAP_75": map_75,
        "mAP_90": map_90,
        "avg_confidence": avg_confidence,
        "detections_above_90_pct": pct_above_90,
        "total_detections": total_detections,
        "detections_above_90": detections_above_90,
        "confidence_distribution": all_confidences,
    }


def compute_map(predictions, targets, iou_threshold=0.5):
    all_tp, all_scores = [], []
    total_gt = 0

    for pred, gt in zip(predictions, targets):
        pred_boxes, pred_scores = pred["boxes"], pred["scores"]
        gt_boxes = gt["boxes"]
        total_gt += len(gt_boxes)

        if len(pred_boxes) == 0:
            continue
        if len(gt_boxes) == 0:
            all_tp.extend([0] * len(pred_boxes))
            all_scores.extend(pred_scores.numpy().tolist())
            continue

        iou_matrix = box_iou(pred_boxes, gt_boxes)
        sorted_idx = torch.argsort(pred_scores, descending=True)
        matched_gt = set()

        for idx in sorted_idx:
            all_scores.append(pred_scores[idx].item())
            ious = iou_matrix[idx]
            max_iou, max_gt_idx = ious.max(0) if len(ious) > 0 else (torch.tensor(0), torch.tensor(0))
            max_gt_idx = max_gt_idx.item()

            if max_iou.item() >= iou_threshold and max_gt_idx not in matched_gt:
                all_tp.append(1)
                matched_gt.add(max_gt_idx)
            else:
                all_tp.append(0)

    if total_gt == 0 or len(all_scores) == 0:
        return 0.0

    sorted_indices = np.argsort(-np.array(all_scores))
    tp = np.array(all_tp)[sorted_indices]
    tp_cumsum = np.cumsum(tp)
    fp_cumsum = np.cumsum(1 - tp)
    recalls = tp_cumsum / total_gt
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)

    # 11-point interpolated AP
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p_at_r = precisions[recalls >= t]
        if len(p_at_r) > 0:
            ap += p_at_r.max()
    return ap / 11.0

print("Evaluation functions defined (mAP@50, mAP@75, mAP@90, confidence analysis)")

In [ ]:
# Training, checkpointing, and prediction visualization

def train_one_epoch(model, optimizer, data_loader, device, epoch, scaler):
    model.train()
    running_losses = defaultdict(float)
    num_batches = 0
    pbar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{config.NUM_EPOCHS}", leave=True)

    for batch_idx, (images, targets) in enumerate(pbar):
        images = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

        if not any(t["boxes"].shape[0] > 0 for t in targets):
            continue

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=config.USE_AMP, dtype=config.AMP_DTYPE):
            loss_dict = model(images, targets)
            total_loss = sum(loss for loss in loss_dict.values())

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        scaler.step(optimizer)
        scaler.update()

        for k, v in loss_dict.items():
            running_losses[k] += v.item()
        running_losses["total"] += total_loss.item()
        num_batches += 1

        mem_gb = torch.cuda.memory_allocated() / 1e9
        pbar.set_postfix({
            "loss": f"{total_loss.item():.4f}",
            "cls": f"{loss_dict.get('loss_classifier', torch.tensor(0)).item():.4f}",
            "box": f"{loss_dict.get('loss_box_reg', torch.tensor(0)).item():.4f}",
            "GPU": f"{mem_gb:.1f}GB"
        })

        global_step = epoch * len(data_loader) + batch_idx
        if batch_idx % 50 == 0:
            tb_writer.add_scalar("batch/total_loss", total_loss.item(), global_step)
            for k, v in loss_dict.items():
                tb_writer.add_scalar(f"batch/{k}", v.item(), global_step)
            tb_writer.add_scalar("batch/gpu_mem_gb", mem_gb, global_step)

    return {k: v / max(num_batches, 1) for k, v in running_losses.items()}


def save_checkpoint(model, optimizer, scheduler, epoch, metrics, filename):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "metrics": {k: v for k, v in metrics.items() if k != "confidence_distribution"},
        "config": {k: v for k, v in vars(config).items()
                   if not k.startswith("_") and not isinstance(v, torch.device)},
    }
    torch.save(checkpoint, filename)

    # Backup to Drive
    drive_path = os.path.join(config.DRIVE_BACKUP_DIR, os.path.basename(filename))
    torch.save(checkpoint, drive_path)
    logger.info(f"Checkpoint saved: {filename} + Drive backup")


@torch.no_grad()
def visualize_predictions(model, train_loader, val_loader, device, epoch, metrics, n=4):
    model.eval()

    # Grab one batch
    train_images, train_targets = next(iter(train_loader))
    val_images, val_targets = next(iter(val_loader))
    train_gpu = [img.to(device, non_blocking=True) for img in train_images]
    val_gpu = [img.to(device, non_blocking=True) for img in val_images]

    with torch.cuda.amp.autocast(enabled=config.USE_AMP, dtype=config.AMP_DTYPE):
        train_outputs = model(train_gpu)
        val_outputs = model(val_gpu)

    n = min(n, len(train_images), len(val_images))
    fig, axes = plt.subplots(3, n, figsize=(5 * n, 15))
    if n == 1:
        axes = axes.reshape(3, 1)

    for i in range(n):
        # Ground truth
        img = np.clip(train_images[i].permute(1, 2, 0).cpu().numpy(), 0, 1)
        axes[0, i].imshow(img)
        gt_boxes = train_targets[i]["boxes"].cpu().numpy()
        for box in gt_boxes:
            xmin, ymin, xmax, ymax = box
            axes[0, i].add_patch(patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                linewidth=2, edgecolor='lime', facecolor='none'
            ))
        axes[0, i].set_title(f"Train GT — {len(gt_boxes)} boxes")
        axes[0, i].axis("off")

        # Predictions on train
        axes[1, i].imshow(img)
        pred_boxes = train_outputs[i]["boxes"].cpu().numpy()
        pred_scores = train_outputs[i]["scores"].cpu().numpy()
        for box, score in zip(pred_boxes, pred_scores):
            if score < config.CONFIDENCE_THRESHOLD:
                continue
            xmin, ymin, xmax, ymax = box
            axes[1, i].add_patch(patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                linewidth=2, edgecolor='yellow', facecolor='none'
            ))
            axes[1, i].text(xmin, ymin - 3, f"{score:.2f}", color='yellow', fontsize=8,
                            fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
        n_pred = (pred_scores >= config.CONFIDENCE_THRESHOLD).sum()
        axes[1, i].set_title(f"Train Pred — {n_pred} boxes")
        axes[1, i].axis("off")

        # Predictions on val
        val_img = np.clip(val_images[i].permute(1, 2, 0).cpu().numpy(), 0, 1)
        axes[2, i].imshow(val_img)
        val_pred_boxes = val_outputs[i]["boxes"].cpu().numpy()
        val_pred_scores = val_outputs[i]["scores"].cpu().numpy()
        for box, score in zip(val_pred_boxes, val_pred_scores):
            if score < config.CONFIDENCE_THRESHOLD:
                continue
            xmin, ymin, xmax, ymax = box
            axes[2, i].add_patch(patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                linewidth=2, edgecolor='yellow', facecolor='none'
            ))
            axes[2, i].text(xmin, ymin - 3, f"{score:.2f}", color='yellow', fontsize=8,
                            fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
        n_val_pred = (val_pred_scores >= config.CONFIDENCE_THRESHOLD).sum()
        axes[2, i].set_title(f"Val Pred — {n_val_pred} boxes")
        axes[2, i].axis("off")

    fig.suptitle(
        f"Epoch {epoch+1} | mAP@50={metrics['mAP_50']:.3f} | mAP@75={metrics['mAP_75']:.3f} | "
        f"mAP@90={metrics['mAP_90']:.3f} | Conf={metrics['avg_confidence']:.3f}",
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()

    save_dir = os.path.join(config.DRIVE_BACKUP_DIR, "prediction_visualizations")
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"best_epoch_{epoch+1:03d}.png")
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Prediction visualization saved: {save_path}")

print("Training, checkpointing, and visualization functions defined")

In [ ]:
# Run training

best_map = 0.0
patience_counter = 0
history = []

for epoch in range(config.NUM_EPOCHS):
    epoch_start = time.time()

    train_losses = train_one_epoch(model, optimizer, train_loader, config.DEVICE, epoch, scaler)
    lr_scheduler.step()
    val_metrics = evaluate_model(model, val_loader, config.DEVICE)

    epoch_time = time.time() - epoch_start
    current_lr = optimizer.param_groups[1]["lr"]

    # TensorBoard
    tb_writer.add_scalar("epoch/train_loss", train_losses["total"], epoch)
    tb_writer.add_scalar("epoch/mAP_50", val_metrics["mAP_50"], epoch)
    tb_writer.add_scalar("epoch/mAP_75", val_metrics["mAP_75"], epoch)
    tb_writer.add_scalar("epoch/mAP_90", val_metrics["mAP_90"], epoch)
    tb_writer.add_scalar("epoch/avg_confidence", val_metrics["avg_confidence"], epoch)
    tb_writer.add_scalar("epoch/lr", current_lr, epoch)

    print(f"\nEpoch {epoch+1}/{config.NUM_EPOCHS} ({epoch_time:.0f}s) | "
          f"Loss: {train_losses['total']:.4f} | "
          f"mAP@50: {val_metrics['mAP_50']:.3f} | mAP@75: {val_metrics['mAP_75']:.3f} | "
          f"mAP@90: {val_metrics['mAP_90']:.3f} | "
          f"Conf: {val_metrics['avg_confidence']:.3f} | LR: {current_lr:.6f}")

    # Save periodic checkpoint
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model, optimizer, lr_scheduler, epoch, val_metrics,
                        os.path.join(config.CHECKPOINT_DIR, f"epoch_{epoch+1:03d}.pth"))

    # On improvement (mAP@75): save best model + visualize predictions
    if val_metrics["mAP_75"] > best_map:
        improvement = val_metrics["mAP_75"] - best_map
        best_map = val_metrics["mAP_75"]
        patience_counter = 0
        print(f"  New best mAP@75: {best_map:.4f} (+{improvement:.4f})")
        save_checkpoint(model, optimizer, lr_scheduler, epoch, val_metrics,
                        os.path.join(config.CHECKPOINT_DIR, "best_model.pth"))
        visualize_predictions(model, train_loader, val_loader, config.DEVICE, epoch, val_metrics, n=4)
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{config.EARLY_STOP_PATIENCE})")

    if patience_counter >= config.EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

    history.append({
        "epoch": epoch + 1, "train_loss": train_losses["total"],
        "mAP_50": val_metrics["mAP_50"], "mAP_75": val_metrics["mAP_75"],
        "mAP_90": val_metrics["mAP_90"], "avg_confidence": val_metrics["avg_confidence"],
        "lr": current_lr, "time": epoch_time,
    })

tb_writer.close()
print(f"\nTraining complete. Best mAP@75: {best_map:.4f}")

In [ ]:
# Load best model checkpoint

checkpoint_path = os.path.join(config.DRIVE_BACKUP_DIR, "best_model.pth")
checkpoint = torch.load(checkpoint_path, map_location=config.DEVICE, weights_only=False)

model = build_model(num_classes=config.NUM_CLASSES, pretrained=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(config.DEVICE)
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")
print(f"  mAP@50: {checkpoint['metrics']['mAP_50']:.3f}")
print(f"  mAP@75: {checkpoint['metrics']['mAP_75']:.3f}")
print(f"  mAP@90: {checkpoint['metrics']['mAP_90']:.3f}")

In [ ]:
# Video inference (cropped to top-left 512x512)

import cv2
from IPython.display import HTML
from base64 import b64encode

VIDEO_INPUT = "/content/drive/MyDrive/debris-detection/simulated_debris_video3.mp4"
VIDEO_OUTPUT = os.path.join(config.DRIVE_BACKUP_DIR, "debris_detection_output.mp4")
CONF_THRESHOLD = 0.97
FRAME_SKIP = 0
CROP_SIZE = 512

@torch.no_grad()
def process_frame(model, frame, device, conf_threshold=0.97):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_tensor = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).to(device)

    with torch.cuda.amp.autocast(enabled=config.USE_AMP, dtype=config.AMP_DTYPE):
        outputs = model([img_tensor])[0]

    boxes = outputs["boxes"].cpu().numpy()
    scores = outputs["scores"].cpu().numpy()

    for box, score in zip(boxes, scores):
        if score < conf_threshold:
            continue
        xmin, ymin, xmax, ymax = box.astype(int)
        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), (0, 255, 255), 2)
        label = f"{score:.2f}"
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (xmin, ymin - h - 8), (xmin + w + 4, ymin), (0, 0, 0), -1)
        cv2.putText(frame, label, (xmin + 2, ymin - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    return frame, (scores >= conf_threshold).sum()


def run_video_inference(model, input_path, output_path, device, conf_threshold=0.97, frame_skip=0, crop_size=512):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"ERROR: Could not open video: {input_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    orig_w, orig_h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    x_start = 0
    y_start = 0
    x_end = min(crop_size, orig_w)
    y_end = min(crop_size, orig_h)
    cw, ch = x_end - x_start, y_end - y_start

    print(f"Original: {orig_w}x{orig_h}, {fps:.1f} FPS, {total_frames} frames")
    print(f"Cropping top-left: [{x_start}:{x_end}, {y_start}:{y_end}] = {cw}x{ch}")

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (cw, ch))
    model.eval()
    frame_idx, processed, total_det = 0, 0, 0
    pbar = tqdm(total=total_frames, desc="Processing video")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        cropped = frame[y_start:y_end, x_start:x_end]

        if frame_skip > 0 and frame_idx % (frame_skip + 1) != 0:
            out.write(cropped)
        else:
            annotated, n_det = process_frame(model, cropped, device, conf_threshold)
            out.write(annotated)
            total_det += n_det
            processed += 1

        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    out.release()
    print(f"Processed {processed}/{total_frames} frames, {total_det} total detections")
    print(f"Saved: {output_path}")

run_video_inference(model, VIDEO_INPUT, VIDEO_OUTPUT, config.DEVICE,
                    conf_threshold=CONF_THRESHOLD, frame_skip=FRAME_SKIP, crop_size=CROP_SIZE)

In [ ]:
# Play annotated video in notebook

def play_video_in_notebook(video_path, width=800):
    if not os.path.exists(video_path):
        print(f"Video not found: {video_path}")
        return

    with open(video_path, "rb") as f:
        video_data = f.read()

    b64 = b64encode(video_data).decode()
    html = f'''
    <video width="{width}" controls>
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    '''
    return HTML(html)

play_video_in_notebook(VIDEO_OUTPUT)

In [ ]:
# Learning curves

import pandas as pd

csv_path = "/content/training_history.csv"
df = pd.read_csv(csv_path)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(df["epoch"], df["loss"], color="red", linewidth=2)
axes[0, 0].set_title("Training Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(True, alpha=0.3)

# mAP
axes[0, 1].plot(df["epoch"], df["mAP_50"], label="mAP@50", linewidth=2)
axes[0, 1].plot(df["epoch"], df["mAP_75"], label="mAP@75", linewidth=2)
axes[0, 1].set_title("mAP Scores")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("mAP")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Confidence
axes[1, 0].plot(df["epoch"], df["conf"], color="green", linewidth=2)
axes[1, 0].set_title("Average Confidence")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Confidence")
axes[1, 0].grid(True, alpha=0.3)

# Learning rate
axes[1, 1].plot(df["epoch"], df["lr"], color="orange", linewidth=2)
axes[1, 1].set_title("Learning Rate")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("LR")
axes[1, 1].set_yscale("log")
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle("Training Learning Curves", fontsize=15, fontweight="bold")
plt.tight_layout()

save_path = os.path.join(config.DRIVE_BACKUP_DIR, "learning_curves.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")